# Single-Market Drift Walkthrough — `KXMLBSPREAD-26MAY101920DETKC-DET2`

First concrete look at the project's headline question — how much does a Kalshi MLB market's price drift between pre-game trading and live (in-game) trading? This notebook works with **N=1 market**: the moneyline-spread `DET2` contract on the May 10 2026 Detroit Tigers @ Kansas City Royals game (YES resolves true if DET wins by 2+ runs).

The goal here is **not** statistical generality — N=1 doesn't get us there. It is to:

1. Demonstrate the data shape end-to-end (market metadata, orderbook snapshot, trade history) for one real example.
2. Establish the analytical primitives (price metrics, time alignment, drift definition) we'll reuse when we scale to many markets.
3. Cache the data to parquet so this notebook renders cleanly on GitHub even when the Snowflake warehouse is paused.

### Prerequisites

This notebook reads from the **dbt staging/marts** (`fct_markets`, `fct_market_orderbooks`, `stg_kalshi_market_trades`), not from `RAW`. If you've just scraped fresh data for this market, run `dbt run` first so the marts pick up the new rows:

```bash
dbt run --project-dir dbt --profiles-dir dbt
```

The first cell run will hit Snowflake once and write parquet to `analysis/data/single_market_KXMLBSPREAD-26MAY101920DETKC-DET2/`. Subsequent re-runs load from that cache and don't need a warm warehouse.

In [4]:
import json
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from dotenv import find_dotenv, load_dotenv

from snow_py.connection import SnowflakeManager

# Walk up from the notebook's cwd to find the repo root .env
load_dotenv(find_dotenv())

MARKET_TICKER = "KXMLBSPREAD-26MAY101920DETKC-DET2"
DATA_DIR = Path("data") / f"single_market_{MARKET_TICKER}"

plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


def cached_query(query: str, name: str, snowflake: SnowflakeManager | None = None) -> pd.DataFrame:
    """Load a Snowflake query result from parquet cache if present, else query and cache.

    Keeps every notebook re-executable without warehouse access as long as the cache
    is on disk — the rendered cell output in the .ipynb is what ships to GitHub.
    """
    cache_path = DATA_DIR / f"{name}.parquet"
    if cache_path.exists():
        print(f"  loaded {name} from cache ({cache_path})")
        return pd.read_parquet(cache_path)
    if snowflake is None:
        raise RuntimeError(
            f"Cache miss for {name} and no Snowflake connection provided. "
            "Pass a SnowflakeManager to populate the cache."
        )
    print(f"  cache miss for {name}; querying Snowflake...")
    rows = snowflake.execute(query)
    df = pd.DataFrame(rows)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    df.to_parquet(cache_path)
    print(f"  wrote {len(df)} rows to {cache_path}")
    return df

In [6]:
# Only open a Snowflake connection if we don't already have all three caches.
need_snowflake = not all(
    (DATA_DIR / f"{name}.parquet").exists()
    for name in ("market", "orderbook", "trades")
)

# Connect to PROD; use fully-qualified table names below so we don't depend on
# the session default schema. Per dbt/macros/generate_schema_name.sql, staging
# models materialize in PROD.STAGE and mart models materialize in PROD.KALSHI.
snowflake = SnowflakeManager("PROD", "STAGE") if need_snowflake else None

market_df = cached_query(
    f"""
    select *
    from PROD.KALSHI.fct_markets
    where market_ticker = '{MARKET_TICKER}'
    """,
    "market",
    snowflake,
)

orderbook_df = cached_query(
    f"""
    select *
    from PROD.KALSHI.fct_market_orderbooks
    where market_ticker = '{MARKET_TICKER}'
    """,
    "orderbook",
    snowflake,
)

trades_df = cached_query(
    f"""
    select *
    from PROD.STAGE.stg_kalshi_market_trades
    where market_ticker = '{MARKET_TICKER}'
    order by trade_time
    """,
    "trades",
    snowflake,
)

if snowflake is not None:
    snowflake.close()

print()
print(f"market    : {len(market_df)} row(s)")
print(f"orderbook : {len(orderbook_df)} row(s)")
print(f"trades    : {len(trades_df)} rows")

INFO:snowflake.connector.connection:Snowflake Connector for Python Version: 4.3.0, Python Version: 3.13.12, Platform: Windows-11-10.0.26200-SP0
INFO:snowflake.connector.connection:Connecting to GLOBAL Snowflake domain


  cache miss for market; querying Snowflake...
  wrote 1 rows to data\single_market_KXMLBSPREAD-26MAY101920DETKC-DET2\market.parquet
  cache miss for orderbook; querying Snowflake...
  wrote 1 rows to data\single_market_KXMLBSPREAD-26MAY101920DETKC-DET2\orderbook.parquet
  cache miss for trades; querying Snowflake...
  wrote 862 rows to data\single_market_KXMLBSPREAD-26MAY101920DETKC-DET2\trades.parquet

market    : 1 row(s)
orderbook : 1 row(s)
trades    : 862 rows


## What this market is

The `fct_markets` row tells us what the contract resolves on, when trading opened and closed, and the latest price snapshot at scrape time. A few Kalshi field-name conventions worth keeping in mind:

- **`_dollars`** suffix — the value is denominated in dollars (e.g. `last_price_dollars = 0.42` means the last trade cleared at 42¢).
- **`_fp`** suffix — Kalshi's "fixed-point" convention; the value is an integer count of contracts with no decimal scaling. `volume_fp` is the lifetime contract count traded.
- **`yes_*` / `no_*`** — two sides of the binary market. YES pays $1 if the market resolves true, NO pays $1 if it resolves false. Prices sum to ~$1 (modulo the bid-ask spread).

In [7]:
assert len(market_df) == 1, f"Expected exactly one market row for {MARKET_TICKER}, got {len(market_df)}"
mkt = market_df.iloc[0]

print(f"Market : {mkt['market_ticker']}")
print(f"Event  : {mkt['event_title']} ({mkt['event_ticker']})")
print(f"Series : {mkt['series_ticker']}")
print(f"Type   : {mkt['market_type']}")
print(f"Status : {mkt['market_status']}")
print()
print("Trading window (UTC-naive per staging model):")
print(f"  opened at : {mkt['open_at']}")
print(f"  closed at : {mkt['close_at']}")
print()
print("Latest snapshot:")
print(f"  last price        : ${mkt['last_price_dollars']}")
print(f"  yes bid / ask     : ${mkt['yes_bid_dollars']} / ${mkt['yes_ask_dollars']}")
print(f"  no  bid / ask     : ${mkt['no_bid_dollars']} / ${mkt['no_ask_dollars']}")
print(f"  liquidity         : ${mkt['liquidity_dollars']}")
print(f"  volume (lifetime) : {mkt['volume_fp']}")
print()
print("Market title :", mkt["market_title"])
print("Yes resolves :", mkt["yes_subtitle"])
print("No  resolves :", mkt["no_subtitle"])

Market : KXMLBSPREAD-26MAY101920DETKC-DET2
Event  : Detroit vs Kansas City: Spread (KXMLBSPREAD-26MAY101920DETKC)
Series : KXMLBSPREAD
Type   : binary
Status : finalized

Trading window (UTC-naive per staging model):
  opened at : 2026-05-10 05:20:00
  closed at : 2026-05-11 02:24:32

Latest snapshot:
  last price        : $0.9900
  yes bid / ask     : $0.9900 / $1.0000
  no  bid / ask     : $0.0000 / $0.0100
  liquidity         : $0.0000
  volume (lifetime) : 82270.800000

Market title : Detroit wins by over 1.5 runs?
Yes resolves : Detroit wins by over 1.5 runs
No  resolves : Detroit wins by over 1.5 runs


## Trade timeline

How did YES price move between `open_at` and `close_at`? Plotting every executed trade with its timestamp. The vertical line marks the assumed game start (encoded in the event ticker as `26MAY101920DETKC` → May 10 2026 19:20 ET = 23:20 UTC). Trade times come from `try_to_timestamp_ntz` and are stored as **naive UTC**, so the boundary is in UTC too; a sanity-check print confirms it lands inside the trade range.

Color by `taker_side` to show which side was lifting the offer at each price point:

- **`yes`** — someone *bought* YES at the ask. Bullish on the outcome.
- **`no`** — someone *bought* NO at the ask, which is equivalent to *selling* YES. Bearish on the outcome.

(YES and NO prices on the same trade sum to ~$1, so plotting only YES gives the full picture.)

In [ ]:
import datetime as dt

from matplotlib.ticker import FormatStrFormatter, MaxNLocator

# Kalshi's `created_time` field arrives as UTC ISO. The staging model uses
# try_to_timestamp_ntz, which strips the offset and stores naive UTC wall-clock.
# GAME_START must therefore also be in UTC (naive). The event ticker encodes
# `19:20` as a wall-clock time without a timezone; defaulting to ET (Kalshi's
# usual convention) -> 19:20 EDT on May 10 2026 = 23:20 UTC. If the sanity-check
# below shows the boundary outside the trade range, adjust (e.g. to CT for a
# 19:20 CDT first pitch -> 00:20 UTC May 11).
GAME_START = dt.datetime(2026, 5, 10, 23, 20)

trades_df["trade_time"] = pd.to_datetime(trades_df["trade_time"])
trades_df = trades_df.sort_values("trade_time").reset_index(drop=True)

# Sanity-check the boundary against the trading window and observed trade range.
print(f"Market open  : {mkt['open_at']}")
print(f"Market close : {mkt['close_at']}")
print(f"Trade range  : {trades_df['trade_time'].min()} -> {trades_df['trade_time'].max()}")
print(f"GAME_START   : {GAME_START}  (assumed 19:20 EDT = 23:20 UTC)")
print(f"YES price    : ${trades_df['yes_price_dollars'].min():.3f} -> ${trades_df['yes_price_dollars'].max():.3f}")

fig, ax = plt.subplots(figsize=(11, 5))

for side, color, label in [
    ("yes", "tab:green", "taker bought YES"),
    ("no", "tab:red", "taker bought NO (sold YES)"),
]:
    subset = trades_df[trades_df["taker_side"] == side]
    ax.scatter(
        subset["trade_time"],
        subset["yes_price_dollars"],
        s=10,
        alpha=0.5,
        c=color,
        label=label,
    )

ax.axvline(GAME_START, color="black", linewidth=1.2, linestyle="--", label="game start (assumed)")
ax.set_xlabel("trade time (UTC)")
ax.set_ylabel("YES price")

# Cap to the [0, 1] payout range so a reader sees the price band relative to the
# two binary outcomes; cap further down only if the range is so narrow that this
# obscures the structure of the moves.
ax.set_ylim(0, 1)
ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
ax.yaxis.set_major_formatter(FormatStrFormatter("$%.2f"))

ax.set_title(f"{MARKET_TICKER} — YES price by trade")
ax.legend(loc="best")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Orderbook snapshot

Unlike trades, the orderbook is a single point-in-time picture of resting orders — a snapshot taken at scrape time, not a time series. The Snowflake `orderbook` column is a JSON `VARIANT` keyed by side:

- `yes_dollars` — a list of `[price, size]` pairs for resting **bids on YES** (someone willing to pay that price for YES).
- `no_dollars` — same for the NO side.

The two sides together approximate the full demand curve at the scrape moment.

In [ ]:
assert len(orderbook_df) == 1, f"Expected exactly one orderbook row for {MARKET_TICKER}, got {len(orderbook_df)}"

# The orderbook column round-trips through Snowflake VARIANT -> JSON string.
orderbook_raw = orderbook_df.iloc[0]["orderbook"]
orderbook = json.loads(orderbook_raw) if isinstance(orderbook_raw, str) else orderbook_raw

# Diagnostic: surface what the dbt model computed and what keys the raw payload
# actually has, so a 0-level result is debuggable rather than mysterious.
print(f"Top-level orderbook keys : {list(orderbook.keys())}")
print(f"dbt yes_level_count      : {orderbook_df.iloc[0]['yes_level_count']}")
print(f"dbt no_level_count       : {orderbook_df.iloc[0]['no_level_count']}")
print(f"dbt has_yes_orders       : {orderbook_df.iloc[0]['has_yes_orders']}")
print(f"dbt has_no_orders        : {orderbook_df.iloc[0]['has_no_orders']}")
print(f"market_status            : {orderbook_df.iloc[0]['market_status']}")
print(f"market_title             : {orderbook_df.iloc[0]['market_title']}")

# Kalshi's orderbook keys are typically `yes` and `no` (price in cents) for the
# `_fp` (fixed-point) payload, but the dbt model in this repo treats them as
# `yes_dollars` / `no_dollars`. Try both so the cell doesn't silently produce an
# empty chart if the key name doesn't match the assumption.
def _first_nonempty(d, keys):
    for k in keys:
        v = d.get(k)
        if v:
            return k, v
    return None, []

yes_key, yes_raw = _first_nonempty(orderbook, ["yes_dollars", "yes"])
no_key, no_raw = _first_nonempty(orderbook, ["no_dollars", "no"])
print(f"Using YES key: {yes_key!r} ({len(yes_raw)} levels)")
print(f"Using NO  key: {no_key!r} ({len(no_raw)} levels)")

yes_levels = pd.DataFrame(yes_raw, columns=["price", "size"]).astype(float)
no_levels = pd.DataFrame(no_raw, columns=["price", "size"]).astype(float)

# If the prices look like cents (any value > 1), divide by 100 to get dollars
# so both sides plot on the same x-axis as the trades chart.
def _to_dollars(df):
    if not df.empty and df["price"].max() > 1:
        df = df.copy()
        df["price"] = df["price"] / 100
    return df

yes_levels = _to_dollars(yes_levels)
no_levels = _to_dollars(no_levels)

fig, ax = plt.subplots(figsize=(10, 4))
if not yes_levels.empty:
    ax.bar(yes_levels["price"], yes_levels["size"], width=0.005, color="tab:green", alpha=0.7, label="YES bids")
if not no_levels.empty:
    ax.bar(no_levels["price"], no_levels["size"], width=0.005, color="tab:red", alpha=0.7, label="NO bids")
ax.set_xlim(0, 1)
ax.set_xlabel("price ($)")
ax.set_ylabel("contracts resting")
ax.set_title(f"{MARKET_TICKER} — orderbook snapshot at scrape time")
if not yes_levels.empty or not no_levels.empty:
    ax.legend()
plt.tight_layout()
plt.show()

print()
print(f"YES side: {len(yes_levels)} price levels totalling {int(yes_levels['size'].sum())} contracts")
print(f"NO  side: {len(no_levels)} price levels totalling {int(no_levels['size'].sum())} contracts")

## First-pass drift metric

Split the trades into **pre-game** and **live** windows using the encoded game start, then compare basic price statistics within each. The point isn't to draw a final conclusion from N=1 — it's to **define what "drift" means** so the same metric scales cleanly to many markets later.

Working definition for this notebook:

- **range** = `max(YES price) − min(YES price)` within the window. Captures the widest swing.
- **std-dev** = `std(YES price)` within the window. Captures volatility around the mean.

A larger live-window range or std-dev than pre-game would be consistent with more new information arriving once the game starts. The directional question — *does drift typically widen or tighten at the bell?* — needs N>>1 markets to answer.

In [ ]:
trade_min = trades_df["trade_time"].min()
trade_max = trades_df["trade_time"].max()

# pd.cut needs strictly increasing bin edges. If GAME_START falls outside the
# observed trade range, all trades end up in one window and the cut breaks —
# usually a sign the timezone assumption is wrong (see plot-trades cell).
if GAME_START <= trade_min:
    raise ValueError(
        f"GAME_START ({GAME_START}) is at/before the first trade ({trade_min}). "
        f"All trades would be classified as 'live'. Adjust the timezone assumption."
    )
if GAME_START >= trade_max:
    raise ValueError(
        f"GAME_START ({GAME_START}) is at/after the last trade ({trade_max}). "
        f"All trades would be classified as 'pre-game'. Adjust the timezone assumption."
    )

trades_df["window"] = pd.cut(
    trades_df["trade_time"],
    bins=[
        trade_min - pd.Timedelta(seconds=1),
        GAME_START,
        trade_max + pd.Timedelta(seconds=1),
    ],
    labels=["pre-game", "live"],
)

summary = (
    trades_df.groupby("window", observed=True)
    .agg(
        n_trades=("trade_id", "count"),
        first_trade=("trade_time", "min"),
        last_trade=("trade_time", "max"),
        yes_price_mean=("yes_price_dollars", "mean"),
        yes_price_std=("yes_price_dollars", "std"),
        yes_price_min=("yes_price_dollars", "min"),
        yes_price_max=("yes_price_dollars", "max"),
    )
    .assign(yes_price_range=lambda d: d["yes_price_max"] - d["yes_price_min"])
)
summary

## Observable findings

_Fill these in after running the cells above. Discipline (see project memory note on narrative drift): describe **what the data shows**, not a causal story for **why** it moved. Causation needs more than one market and more than one game._

- N trades pre-game: …
- N trades live: …
- YES price range pre-game: $… → $…
- YES price range live: $… → $…
- Std-dev ratio (live / pre-game): …

## What this unlocks

This notebook is the first lap. To get from here to a defensible answer on the headline question:

1. **More markets.** At least one full slate (~15 games × ~5 market types per game = ~75 markets) before pre-game vs. live differences are anything beyond anecdote.
2. **A real `pre-game` boundary.** Game start time should live in a `dim_events` enrichment, not be parsed from the event ticker. That is the next dbt-side piece.
3. **Liquidity context.** The headline question pairs drift with **liquidity**. Need to bring `liquidity_dollars` and `yes_bid_size_fp` / `yes_ask_size_fp` into the comparison once N>1.

Subsequent notebooks (`02_*`, `03_*`) will pick these up.